Lora Config:

```
lora_cfg = LoraConfig(
        task_type       = TaskType.CAUSAL_LM,
        r               = 8,
        lora_alpha      = 16,
        lora_dropout    = LORA_DROPOUT,
        target_modules  = LORA_TARGET_MODULES,
        bias            = "none",
    )
```

**Wandb:**

  - https://wandb.ai/kunjcr2-dreamable/adaptroute-adapters/workspace?nw=nwuserkunjcr2

**Huggingface:**

  - https://huggingface.co/kunjcr2/code-adaptroute
  - https://huggingface.co/kunjcr2/math-adaptroute
  - https://huggingface.co/kunjcr2/qa-adaptroute
  - https://huggingface.co/kunjcr2/medical-adaptroute



---

Lora Config:

```
lora_cfg = LoraConfig(
        task_type       = TaskType.CAUSAL_LM,
        r               = 16,
        lora_alpha      = 32,
        lora_dropout    = LORA_DROPOUT,
        target_modules  = LORA_TARGET_MODULES,
        bias            = "none",
    )
```

**Wandb:**

  - https://wandb.ai/kunjcr2-dreamable/adaptroute-adapters-v2?nw=nwuserkunjcr2

**Huggingface:**

  - https://huggingface.co/kunjcr2/code-adaptroute-v2
  - https://huggingface.co/kunjcr2/math-adaptroute-v2
  - https://huggingface.co/kunjcr2/qa-adaptroute-v2
  - https://huggingface.co/kunjcr2/medical-adaptroute-v2

# AdaptRoute — LoRA Adapter Training (All 4 Domains)

Trains one LoRA adapter per domain on top of a **frozen, 4-bit NF4 quantised Qwen2.5-1.5B** base model,
then pushes each adapter to HuggingFace Hub.

**New architecture:** `Query → Firewall → Gate → Adapter merge → Response`

| Domain | HF Dataset | Adapter pushed as |
|--------|-----------|-------------------|
| code | `iamtarun/python_code_instructions_18k_alpaca` | `{HF_USERNAME}/code-adaptroute` |
| math | `lighteval/MATH` (via DigitalLearningGmbH/MATH-lighteval) | `{HF_USERNAME}/math-adaptroute` |
| qa | `rajpurkar/squad` | `{HF_USERNAME}/qa-adaptroute` |
| medical | `lavita/ChatDoctor-HealthCareMagic-100k` | `{HF_USERNAME}/medical-adaptroute` |

**All 4 adapters trained sequentially in a single run.  
Expected runtime on A100:** ~25 min per adapter (~100 min total)

In [1]:
!pip install -U bitsandbytes>=0.46.1 trl datasets pyarrow

## 1. Configuration
Set `HF_USERNAME`. Everything else is pre-tuned for Qwen2.5-1.5B on A100.

In [2]:
# ─── USER CONFIG ──────────────────────────────────────────────────────────────
HF_USERNAME   = "kunjcr2"
WANDB_PROJECT = "adaptroute-adapters-v2"  # set None to skip W&B

BASE_MODEL    = "Qwen/Qwen2.5-1.5B"
N_SAMPLES     = 20_000   # samples per domain
MAX_LENGTH    = 1024      # token budget per example
OUTPUT_ROOT   = "./adapters"
SEED          = 42

# LoRA — same config applied to all 4 adapters
LORA_R               = 16
LORA_ALPHA           = 32
LORA_DROPOUT         = 0.05
LORA_TARGET_MODULES  = ["q_proj", "k_proj", "v_proj", "o_proj"]  # Qwen2 attention

# SFT training
BATCH_SIZE      = 32
GRAD_ACCUM      = 4
NUM_EPOCHS      = 3
LEARNING_RATE   = 2e-4
WARMUP_RATIO    = 0.05
# ──────────────────────────────────────────────────────────────────────────────

# Adapter configs — each entry drives a full train-and-push cycle
ADAPTERS = [
    {
        "name"       : "code",
        "hf_dataset" : "iamtarun/python_code_instructions_18k_alpaca",
        "hf_config"  : None,
        "hf_split"   : "train",
        "stream"     : False,
    },
    {
        "name"       : "math",
        "hf_dataset" : "DigitalLearningGmbH/MATH-lighteval",  # drop-in for lighteval/MATH
        "hf_config"  : None,
        "hf_split"   : "train",
        "stream"     : False,
    },
    {
        "name"       : "qa",
        "hf_dataset" : "rajpurkar/squad",
        "hf_config"  : None,
        "hf_split"   : "train",
        "stream"     : False,
    },
    {
        "name"       : "medical",
        "hf_dataset" : "lavita/ChatDoctor-HealthCareMagic-100k",
        "hf_config"  : None,
        "hf_split"   : "train",
        "stream"     : False,
    },
]

import os
os.makedirs(OUTPUT_ROOT, exist_ok=True)
print(f"Base model : {BASE_MODEL}")
print(f"Adapters   : {[a['name'] for a in ADAPTERS]}")
print(f"Will push  : {[HF_USERNAME + '/' + a['name'] + '-adaptroute' for a in ADAPTERS]}")

Base model : Qwen/Qwen2.5-1.5B
Adapters   : ['code', 'math', 'qa', 'medical']
Will push  : ['kunjcr2/code-adaptroute', 'kunjcr2/math-adaptroute', 'kunjcr2/qa-adaptroute', 'kunjcr2/medical-adaptroute']


## 2. Authentication

In [3]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN, add_to_git_credential=False)
print("✓ HuggingFace login OK")

✓ HuggingFace login OK


In [4]:
if WANDB_PROJECT:
    import wandb
    wandb.login(key=userdata.get("WANDB_API"))
    print("✓ W&B ready (runs will be started per-adapter)")
else:
    print("Skipping W&B")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kunjcr2 (kunjcr2-dreamable) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✓ W&B ready (runs will be started per-adapter)


## 3. Dataset Formatters

Each raw dataset has a different schema. These functions normalise every domain
into a single Qwen2.5 chat-formatted string:

```
<|im_start|>user
{instruction}<|im_end|>
<|im_start|>assistant
{response}<|im_end|>
```

| Domain | Raw fields used | Mapping |
|--------|----------------|--------|
| code | `instruction`, `input`, `output` | instruction + input as user turn, output as response |
| math | `problem`, `solution` | instruction = problem, response = solution |
| qa | `context`, `question`, `answers.text[0]` | instruction = context + question, response = answer |
| medical | `instruction`, `input`, `output` | instruction = instruction + input, response = output |

In [5]:
def fmt_qwen(instruction: str, response: str) -> str:
    """Wrap a single instruction-response pair in Qwen2.5 chat tokens."""
    return (
        f"<|im_start|>user\n{instruction.strip()}<|im_end|>\n"
        f"<|im_start|>assistant\n{response.strip()}<|im_end|>"
    )

# ── Per-domain formatters ────────────────────────────────────────────────────
def format_code(record: dict) -> str | None:
    """
    iamtarun/python_code_instructions_18k_alpaca:
    {instruction, input, output, prompt}

    The `prompt` column already concatenates instruction + input + output
    in Alpaca format. We extract instruction+input as the user turn and
    output as the assistant turn, then wrap in Qwen2.5 chat tokens.
    """
    instruction = record.get("instruction", "").strip()
    inp         = record.get("input",       "").strip()
    output      = record.get("output",      "").strip()
    if not instruction or not output:
        return None
    user_turn = f"{instruction}\n{inp}" if inp else instruction
    return fmt_qwen(user_turn, output)

def format_math(record: dict) -> str | None:
    """
    DigitalLearningGmbH/MATH-lighteval: {problem, level, type, solution}
    Direct mapping — problem is the instruction, solution is the response.
    """
    problem  = record.get("problem",  "").strip()
    solution = record.get("solution", "").strip()
    if not problem or not solution:
        return None
    return fmt_qwen(problem, solution)

def format_qa(record: dict) -> str | None:
    """
    rajpurkar/squad: {context, question, answers: {text: [...], answer_start: [...]}}
    Formats as a grounded reading-comprehension task.
    """
    context  = record.get("context",  "").strip()
    question = record.get("question", "").strip()
    answers  = record.get("answers",  {})
    texts    = answers.get("text", [])
    if not context or not question or not texts:
        return None
    answer      = texts[0].strip()
    instruction = f"Read the following passage and answer the question.\n\nPassage:\n{context}\n\nQuestion: {question}"
    return fmt_qwen(instruction, answer)

def format_medical(record: dict) -> str | None:
    """
    lavita/ChatDoctor-HealthCareMagic-100k: {instruction, input, output}
    Alpaca-style — instruction + input merged as the user turn.
    """
    instruction = record.get("instruction", "").strip()
    inp         = record.get("input",       "").strip()
    output      = record.get("output",      "").strip()
    if not output:
        return None
    user_turn = f"{instruction}\n{inp}" if inp else instruction
    return fmt_qwen(user_turn, output)

FORMATTERS = {
    "code"    : format_code,
    "math"    : format_math,
    "qa"      : format_qa,
    "medical" : format_medical,
}

print("✓ Formatters defined")

# ── Quick sanity-check with a dummy record ───────────────────────────────────
dummy_math = {"problem": "What is 2+2?", "solution": "4", "level": "Level 1", "type": "Algebra"}
print("\nSample math record:")
print(format_math(dummy_math))

✓ Formatters defined

Sample math record:
<|im_start|>user
What is 2+2?<|im_end|>
<|im_start|>assistant
4<|im_end|>


## 4. Load Base Model (once, shared across all adapters)

The base model is loaded **once** in 4-bit NF4 and kept frozen throughout.
Each adapter training run starts fresh from this same frozen base.

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit               = True,
    bnb_4bit_quant_type        = "nf4",
    bnb_4bit_compute_dtype     = torch.bfloat16,
    bnb_4bit_use_double_quant  = True,
)

print(f"Loading {BASE_MODEL} in 4-bit NF4 ...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config = bnb_config,
    device_map          = "auto",
    trust_remote_code   = True,
    torch_dtype         = torch.bfloat16,
)
base_model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # required for causal LM training

total_params = sum(p.numel() for p in base_model.parameters())
print(f"✓ Loaded | Total params: {total_params/1e9:.2f}B")

Loading Qwen/Qwen2.5-1.5B in 4-bit NF4 ...


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✓ Loaded | Total params: 0.89B


## 5. Training Loop — All 4 Adapters

For each domain:
1. Stream / load the raw HF dataset
2. Format records using the domain formatter
3. Attach a fresh LoRA adapter to the frozen base
4. Run SFT with `SFTTrainer`
5. Save adapter locally
6. Push to `{HF_USERNAME}/{domain}-adaptroute`
7. Detach the adapter so the base is clean for the next domain

In [7]:
import gc
import random
import json
from pathlib import Path

import wandb
from datasets import load_dataset, Dataset
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from huggingface_hub import HfApi
from transformers import TrainingArguments

random.seed(SEED)
api = HfApi()


def load_and_format(adapter_cfg: dict, formatter, n: int) -> Dataset:
    """Load raw HF data, apply formatter, return a Dataset of {text: str}."""
    name    = adapter_cfg["name"]
    hf_ds   = adapter_cfg["hf_dataset"]
    cfg     = adapter_cfg["hf_config"]
    split   = adapter_cfg["hf_split"]
    stream  = adapter_cfg["stream"]

    print(f"  Loading {hf_ds} (streaming={stream}) ...")

    if stream:
        raw = load_dataset(hf_ds, name=cfg, split=split, streaming=True,
                           trust_remote_code=True)
        records = []
        for rec in raw:
            if len(records) >= n * 3:
                break
            records.append(rec)
    else:
        kwargs = {"split": split, "trust_remote_code": True}
        if cfg:
            kwargs["name"] = cfg
        raw     = load_dataset(hf_ds, **kwargs)
        records = list(raw)

    random.shuffle(records)

    texts = []
    for rec in records:
        formatted = formatter(rec)
        if formatted and len(formatted) > 50:
            texts.append({"text": formatted})
        if len(texts) >= n:
            break

    if len(texts) < n:
        print(f"  ⚠️  Only {len(texts)} valid samples found (requested {n}).")
    else:
        print(f"  ✓ {len(texts)} samples formatted")

    return Dataset.from_list(texts)


def train_adapter(adapter_cfg: dict) -> None:
    name       = adapter_cfg["name"]
    repo_id    = f"{HF_USERNAME}/{name}-adaptroute-v2"
    output_dir = f"{OUTPUT_ROOT}/{name}"
    formatter  = FORMATTERS[name]

    print(f"\n{'='*65}")
    print(f"  TRAINING: {name.upper()} adapter  →  {repo_id}")
    print(f"{'='*65}")

    # 1. Data
    dataset = load_and_format(adapter_cfg, formatter, N_SAMPLES)

    # 2. Fresh LoRA adapter on the frozen base
    lora_cfg = LoraConfig(
        task_type       = TaskType.CAUSAL_LM,
        r               = LORA_R,
        lora_alpha      = LORA_ALPHA,
        lora_dropout    = LORA_DROPOUT,
        target_modules  = LORA_TARGET_MODULES,
        bias            = "none",
    )
    model = get_peft_model(base_model, lora_cfg)
    model.print_trainable_parameters()

    # 3. W&B run (optional)
    if WANDB_PROJECT:
        wandb.init(project=WANDB_PROJECT, name=f"{name}-adapter", reinit=True)

    # 4. SFT
    total_steps  = (len(dataset) // (BATCH_SIZE * GRAD_ACCUM)) * NUM_EPOCHS
    warmup_steps = max(1, int(total_steps * WARMUP_RATIO))

    sft_config = SFTConfig(
        output_dir                  = output_dir,
        num_train_epochs            = NUM_EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        learning_rate               = LEARNING_RATE,
        warmup_steps                = warmup_steps,
        lr_scheduler_type           = "cosine",
        bf16                        = True,
        logging_steps               = 10,
        save_strategy               = "epoch",
        report_to                   = "wandb" if WANDB_PROJECT else "none",
        dataset_text_field          = "text",
        max_length                  = MAX_LENGTH,
        packing                     = True,   # pack short sequences for efficiency
        seed                        = SEED,
    )

    trainer = SFTTrainer(
        model      = model,
        args       = sft_config,
        train_dataset = dataset,
        processing_class  = tokenizer,
    )

    trainer.train()

    # 5. Save adapter only (not the full model)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"  ✓ Adapter saved locally → {output_dir}")

    # 6. Write model card
    card = f"""---
base_model: {BASE_MODEL}
library_name: peft
license: apache-2.0
tags:
  - lora
  - peft
  - adaptroute
  - {name}
---

# {name}-adaptroute

LoRA adapter for the **{name}** domain in [AdaptRoute](https://adaptroute.vercel.app).

Mounted onto a frozen 4-bit NF4 quantised `{BASE_MODEL}` at inference time via
`peft.add_weighted_adapter()` — weights provided by the gating network.

## LoRA Config
- r = {LORA_R}, alpha = {LORA_ALPHA}, dropout = {LORA_DROPOUT}
- Target modules: {LORA_TARGET_MODULES}
- Training: {NUM_EPOCHS} epochs on {N_SAMPLES} samples, lr={LEARNING_RATE}

## Training Data
- Source: `{adapter_cfg['hf_dataset']}`
"""
    Path(f"{output_dir}/README.md").write_text(card)

    # 7. Push adapter to Hub
    api.create_repo(repo_id=repo_id, exist_ok=True, token=HF_TOKEN)
    api.upload_folder(
        folder_path    = output_dir,
        repo_id        = repo_id,
        token          = HF_TOKEN,
        commit_message = f"Add {name}-adaptroute LoRA adapter",
    )
    print(f"  ✓ Pushed → https://huggingface.co/{repo_id}-v2")

    # 8. Detach LoRA so base_model is clean for next domain
    if WANDB_PROJECT:
        wandb.finish()
    del model, trainer, dataset
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  ✓ VRAM cleared\n")


print("✓ Training functions defined — run the next cell to start")

✓ Training functions defined — run the next cell to start


## 6. Run — Train All 4 Adapters

This single cell trains and pushes all adapters sequentially.
To train only one domain, replace `ADAPTERS` with e.g. `[ADAPTERS[0]]`.

In [8]:
for adapter_cfg in ADAPTERS:
    train_adapter(adapter_cfg)

print("\n" + "="*65)
print("ALL ADAPTERS TRAINED AND PUSHED")
print("="*65)
for a in ADAPTERS:
    print(f"  https://huggingface.co/{HF_USERNAME}/{a['name']}-adaptroute")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'iamtarun/python_code_instructions_18k_alpaca' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'iamtarun/python_code_instructions_18k_alpaca' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.



  TRAINING: CODE adapter  →  kunjcr2/code-adaptroute-v2
  Loading iamtarun/python_code_instructions_18k_alpaca (streaming=False) ...


README.md:   0%|          | 0.00/905 [00:00<?, ?B/s]

data/train-00000-of-00001-8b6e212f3e1ece(…):   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18612 [00:00<?, ? examples/s]

...

  ...adapter_model.safetensors:   1%|1         | 90.0kB / 8.75MB            

  ...eckpoint-166/optimizer.pt:   1%|1         |  181kB / 17.6MB            

  ✓ Pushed → https://huggingface.co/kunjcr2/medical-adaptroute-v2


train/entropy,█▅▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁
train/epoch,▁▁▂▂▃▃▄▄▅▅▅▆▆▇▇██
train/global_step,▁▁▂▂▃▃▄▄▅▅▅▆▆▇▇██
train/grad_norm,█▅▃▁▂▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▅███▇▇▆▅▅▄▃▃▂▂▁▁
train/loss,█▅▃▃▂▂▁▁▁▁▁▁▁▁▁▁
train/mean_token_accuracy,▁▅▆▆▇▇▇▇█▇████▇▇█
train/num_tokens,▁▁▂▂▃▃▄▄▅▅▅▆▆▇▇██
total_flos,8.076925406042112e+16
train/entropy,2.45865
train/epoch,2


  ✓ VRAM cleared


ALL ADAPTERS TRAINED AND PUSHED
  https://huggingface.co/kunjcr2/code-adaptroute
  https://huggingface.co/kunjcr2/math-adaptroute
  https://huggingface.co/kunjcr2/qa-adaptroute
  https://huggingface.co/kunjcr2/medical-adaptroute


## 7. Verify — Smoke Test All Adapters from Hub

Loads each adapter back from the Hub and runs a forward pass to confirm
they mount correctly on the base model.

In [12]:
from peft import PeftModel

# More challenging prompts to test the limits of the specialized adapters
VERIFY_PROMPTS = {
    "code": "Write a Python function that takes a list of dictionaries (representing products with 'name', 'price', and 'category') and returns the name of the most expensive product in a specific category. Handle the case where the category is empty.",
    "math": "A tank has two pipes. Pipe A fills the tank in 5 hours, and Pipe B empties it in 8 hours. If both pipes are opened at the same time when the tank is empty, how long will it take to fill the tank to 75% capacity? Show your steps.",
    "qa": "Read the following passage and answer the question.\n\nPassage:\nQuantum entanglement is a physical phenomenon that occurs when a group of particles are generated, interact, or share spatial proximity in a way such that the quantum state of each particle of the group cannot be described independently of the state of the others, including when the particles are separated by a large distance. The topic of quantum entanglement is at the heart of the disparity between classical and quantum physics: entanglement is a primary feature of quantum mechanics lacking in classical mechanics.\n\nQuestion: Based on the passage, what is the fundamental difference between classical and quantum physics regarding the description of particle states?",
    "medical": "Patient Case: A 55-year-old male presents with a sudden onset of 'the worst headache of my life', accompanied by photophobia and neck stiffness. He has no prior history of migraines. What is the most likely differential diagnosis, and what immediate diagnostic steps should be taken?",
}

print("Verifying adapters from Hub with tougher prompts ...\n")

for a in ADAPTERS:
    name    = a["name"]
    # Ensure we point to the latest -v2 versions if that's what was pushed
    repo_id = f"{HF_USERNAME}/{name}-adaptroute-v2"
    prompt  = VERIFY_PROMPTS[name]

    print(f"── {name.upper()} ({repo_id}) ──")
    try:
        # Load the adapter onto the frozen base_model
        peft_model = PeftModel.from_pretrained(base_model, repo_id)
        peft_model.eval()

        enc = tokenizer(
            f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n",
            return_tensors="pt",
        ).to(base_model.device)

        with torch.no_grad():
            out = peft_model.generate(
                **enc,
                max_new_tokens = 200, # Increased for tougher questions
                do_sample      = False,
                temperature    = None,
                top_p          = None,
            )

        response = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
        # Clean up common hallucinations/headers if needed
        response = response.replace("ChatDoctor", "AdaptRoute").replace("Chat Doctor", "AdaptRoute")

        print(f"Prompt  : {prompt}...")
        print(f"Response:\n{response}")
        print(f"✓ OK\n")

        # Cleanup VRAM for the next adapter
        del peft_model
        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"✗ FAILED: {e}\n")

print("Verification complete.")

Verifying adapters from Hub with tougher prompts ...

── CODE (kunjcr2/code-adaptroute-v2) ──
Prompt  : Write a Python function that takes a list of dictionaries (representing products with 'name', 'price', and 'category') and returns the name of the most expensive product in a specific category. Handle the case where the category is empty....
Response:
def most_expensive_product(products, category):
    if not category:
        return None
    max_price = 0
    max_product = None
    for product in products:
        if product['category'] == category:
            if product['price'] > max_price:
                max_price = product['price']
                max_product = product
    return max_product['name'] if max_product else None

products = [
    {'name': 'Product A', 'price': 10, 'category': 'Electronics'},
    {'name': 'Product B', 'price': 20, 'category': 'Electronics'},
    {'name': 'Product C', 'price': 30, 'category': 'Clothing'},
    {'name': 'Product D', 'price': 40, 'categ

## 8. Soft Merge Helper (AdaptRoute inference)

This is the function your main pipeline calls after the gate returns weights.
Paste it into `adapters/merge_adapters.py`.

In [10]:
from peft import PeftModel


def load_adapter(base, repo_id: str):
    """Load a single LoRA adapter from the Hub onto the base model."""
    return PeftModel.from_pretrained(base, repo_id)


def soft_merge_and_generate(
    base_model,
    tokenizer,
    gate_result: dict,
    max_new_tokens: int = 256,
) -> str:
    """
    Takes the output of gate() and runs a soft-merged forward pass.

    gate_result expected shape:
      {
        'blocked'      : False,
        'top_adapters' : ['lora-code', 'lora-math'],   # adapter names
        'top_weights'  : [0.72, 0.28],                 # normalised
        'hard_routed'  : False,
        'probs'        : {...}
      }

    Adapter name → HF repo mapping:
      lora-{domain} → {HF_USERNAME}/{domain}-adaptroute
    """
    if gate_result["blocked"]:
        return "[BLOCKED] This request was identified as a prompt injection attempt."

    query        = gate_result.get("query", "")   # attach query to gate_result upstream
    top_adapters = gate_result["top_adapters"]     # e.g. ['lora-code', 'lora-math']
    top_weights  = gate_result["top_weights"]

    # Map lora-{domain} → HF repo IDs
    adapter_repo_ids = [
        f"{HF_USERNAME}/{a.replace('lora-', '')}-adaptroute"
        for a in top_adapters
    ]

    if len(adapter_repo_ids) == 1 or gate_result.get("hard_routed"):
        # Hard routing — single adapter, no merge needed
        model = PeftModel.from_pretrained(base_model, adapter_repo_ids[0])
    else:
        # Soft routing — load first adapter, then merge in remaining with weights
        model = PeftModel.from_pretrained(
            base_model, adapter_repo_ids[0], adapter_name=top_adapters[0]
        )
        for repo_id, name in zip(adapter_repo_ids[1:], top_adapters[1:]):
            model.load_adapter(repo_id, adapter_name=name)

        model.add_weighted_adapter(
            adapters         = top_adapters,
            weights          = top_weights,
            adapter_name     = "merged",
            combination_type = "linear",
        )
        model.set_adapter("merged")

    model.eval()
    prompt = f"<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n"
    enc    = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens = max_new_tokens,
            do_sample      = False,
            temperature    = None,
            top_p          = None,
        )

    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)


print("✓ soft_merge_and_generate() defined")
print("  Copy this cell into adapters/merge_adapters.py for use in the main pipeline.")

✓ soft_merge_and_generate() defined
  Copy this cell into adapters/merge_adapters.py for use in the main pipeline.
